# GSM8K GRPO training on Colab

This notebook runs the `RL_GRPO_train` pipeline from Google Drive. It defaults to a 1-step smoke run; set `RUN_SMOKE = False` to launch the YAML configuration as-is.

In [33]:
!nvidia-smi

Mon May  4 20:41:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   31C    P0             47W /  600W |       3MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [34]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [35]:
from pathlib import Path
import os

candidates = [
    Path('/content/drive/MyDrive/HPML_project'),
    Path('/content/drive/MyDrive/HPML_project/project'),
    Path.cwd(),
]
PROJECT_DIR = next((p for p in candidates if (p / 'RL_GRPO_train').exists() and (p / 'RL_common').exists()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError('Cannot find project root. Mount Drive or edit PROJECT_DIR manually.')

os.chdir(PROJECT_DIR)
print('PROJECT_DIR =', PROJECT_DIR)
!pwd

PROJECT_DIR = /content/drive/MyDrive/HPML_project
/content/drive/MyDrive/HPML_project


## Tokens and SwanLab

For VSCode Colab extension, Colab Secrets may be unavailable. The default path below uses `getpass` and environment variables. If you run in the Colab web UI and want to use Secrets, set `USE_COLAB_SECRETS = True`.

In [36]:
import getpass
import os

USE_COLAB_SECRETS = True

if USE_COLAB_SECRETS:
    from google.colab import userdata
    for key in ['HF_TOKEN', 'SWANLAB_API_KEY']:
        value = userdata.get(key)
        if value:
            os.environ[key] = value
else:
    for key in ['HF_TOKEN', 'SWANLAB_API_KEY']:
        if not os.environ.get(key):
            value = getpass.getpass(f'{key} (press Enter to skip): ')
            if value:
                os.environ[key] = value

os.environ.setdefault('SWANLAB_PROJECT', 'gsm8k-grpo')
# Optional: set this if your SwanLab account uses a specific workspace.
# os.environ.setdefault('SWANLAB_WORKSPACE', 'your-workspace')

if os.environ.get('HF_TOKEN'):
    from huggingface_hub import login
    login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)

print('SWANLAB_PROJECT =', os.environ.get('SWANLAB_PROJECT'))
print('HF_TOKEN set =', bool(os.environ.get('HF_TOKEN')))
print('SWANLAB_API_KEY set =', bool(os.environ.get('SWANLAB_API_KEY')))

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


SWANLAB_PROJECT = gsm8k-grpo
HF_TOKEN set = True
SWANLAB_API_KEY set = True


## Install dependencies

In [37]:
!pip uninstall -y torchao
!pip install -q -U "trl[vllm]>=0.29.0" "accelerate>=1.4.0" swanlab bitsandbytes sentencepiece
!pip install -q -e RL_common

import torch
print('torch =', torch.__version__)
print('cuda available =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu =', torch.cuda.get_device_name(0))

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for rl-common (pyproject.toml) ... done
torch = 2.10.0+cu128
cuda available = True
gpu = NVIDIA RTX PRO 6000 Blackwell Server Edition


In [38]:
import torch
import trl
import vllm
import accelerate

print("torch", torch.__version__)
print("trl", trl.__version__)
print("vllm import ok")
print("cuda", torch.cuda.is_available())


torch 2.10.0+cu128
trl 1.3.0
vllm import ok
cuda True


## Select config

`RUN_SMOKE=True` creates a tiny temporary config for checking the trainer, rewards, SwanLab logging, and best-adapter saving. For the real run, set it to `False` and edit the YAML directly.

In [39]:
import yaml
from pathlib import Path

BASE_CONFIG = PROJECT_DIR / 'RL_GRPO_train/configs/qwen25_3b_sft_grpo.yaml'
RUN_SMOKE = False

if RUN_SMOKE:
    cfg = yaml.safe_load(BASE_CONFIG.read_text(encoding='utf-8'))
    cfg['run']['name'] = cfg['run']['name'] + '_smoke'
    cfg['train_dataset']['limit'] = 8
    cfg['eval_dataset']['limit'] = 8
    cfg['eval']['every_steps'] = 1
    cfg['eval']['batch_size'] = 4
    cfg['eval']['max_new_tokens'] = 128
    cfg['reward']['max_completion_tokens'] = 128
    cfg['grpo']['max_steps'] = 1
    cfg['grpo']['logging_steps'] = 1
    cfg['grpo']['num_generations'] = 2
    cfg['grpo']['generation_batch_size'] = 4
    cfg['grpo']['per_device_train_batch_size'] = 2
    cfg['grpo']['gradient_accumulation_steps'] = 1
    cfg['grpo']['max_completion_length'] = 128
    cfg['grpo']['run_name'] = cfg['grpo']['run_name'] + '_smoke'
    ACTIVE_CONFIG = Path('/content/grpo_smoke.yaml')
    ACTIVE_CONFIG.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding='utf-8')
else:
    ACTIVE_CONFIG = BASE_CONFIG

print('ACTIVE_CONFIG =', ACTIVE_CONFIG)
print(ACTIVE_CONFIG.read_text(encoding='utf-8')[:2000])

ACTIVE_CONFIG = /content/drive/MyDrive/HPML_project/RL_GRPO_train/configs/qwen25_3b_sft_grpo.yaml
run:
  name: qwen25_3b_instruct_sft_grpo_g{grpo.num_generations}_train{train_dataset.limit}
  output_root: RL_GRPO_train/outputs

model:
  base_model_name_or_path: Qwen/Qwen2.5-3B-Instruct
  adapter_path: SFT_train/outputs/qwen25_3b_gsm8k_lora_sft_full
  tokenizer_name_or_path: Qwen/Qwen2.5-3B-Instruct
  trust_remote_code: true
  prompt_template: qwen_chat
  system_prompt: ""
  include_empty_system: false
  dtype: bf16
  device_map:
  load_in_4bit: false
  is_trainable_adapter: true

train_dataset:
  kind: sft_train
  name: openai/gsm8k
  config: main
  path: SFT_train/data/gsm8k_sft_train.json
  limit:

eval_dataset:
  kind: sft_val
  name: openai/gsm8k
  config: main
  path: SFT_train/data/gsm8k_sft_val.json
  limit: 725

final_eval_dataset:
  kind: gsm8k_test
  name: openai/gsm8k
  config: main
  start_index: 0
  limit:

reward:
  answer_correct: 1.0
  answer_incorrect: 0.0
  format_rew

## Train

Outputs are written under `RL_GRPO_train/outputs/{resolved_run_name}`. Training metrics go to SwanLab through `report_to: swanlab`; greedy validation metrics are logged as `sft_val/*` by the callback.

In [40]:
!accelerate launch --num_processes 1 RL_GRPO_train/train_grpo.py --config "{ACTIVE_CONFIG}"

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
2026-05-04 20:42:34.861074: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-04 20:42:34.894195: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
`torch_dtype` i

## Inspect outputs

In [41]:
import json
import sys
from pathlib import Path

COMMON_SRC = Path("/content/drive/MyDrive/HPML_project/RL_common/src")
if str(COMMON_SRC) not in sys.path:
    sys.path.insert(0, str(COMMON_SRC))

from rl_common.config import format_run_name, load_config, make_run_dir

cfg = load_config(str(ACTIVE_CONFIG))
cfg['run']['resolved_name'] = format_run_name(cfg['run']['name'], cfg)
run_dir = make_run_dir(cfg['run']['output_root'], cfg['run']['resolved_name'])
print('run_dir =', run_dir)
!find "{run_dir}" -maxdepth 2 -type f | sort

summary_path = run_dir / 'train_summary.json'
if summary_path.exists():
    print(summary_path.read_text(encoding='utf-8'))

best_path = run_dir / 'best_eval_results.json'
if best_path.exists():
    data = json.loads(best_path.read_text(encoding='utf-8'))
    print(json.dumps(data['summary'], indent=2, ensure_ascii=False))

run_dir = /content/drive/MyDrive/HPML_project/RL_GRPO_train/outputs/qwen25_3b_instruct_sft_grpo_g16_trainall
/content/drive/MyDrive/HPML_project/RL_GRPO_train/outputs/qwen25_3b_instruct_sft_grpo_g16_trainall/best_eval_results.json
/content/drive/MyDrive/HPML_project/RL_GRPO_train/outputs/qwen25_3b_instruct_sft_grpo_g16_trainall/final_adapter/adapter_config.json
/content/drive/MyDrive/HPML_project/RL_GRPO_train/outputs/qwen25_3b_instruct_sft_grpo_g16_trainall/final_adapter/adapter_model.safetensors
/content/drive/MyDrive/HPML_project/RL_GRPO_train/outputs/qwen25_3b_instruct_sft_grpo_g16_trainall/final_adapter/README.md
/content/drive/MyDrive/HPML_project/RL_GRPO_train/outputs/qwen25_3b_instruct_sft_grpo_g16_trainall/latest_eval_results.json
/content/drive/MyDrive/HPML_project/RL_GRPO_train/outputs/qwen25_3b_instruct_sft_grpo_g16_trainall/resolved_config.yaml
/content/drive/MyDrive/HPML_project/RL_GRPO_train/outputs/qwen25_3b_instruct_sft_grpo_g16_trainall/train_summary.json
{
  "global_

## Optional final eval on official GSM8K test

Do not run this during training. This cell rewrites a temporary eval config so `adapter_path` points to the trained `final_adapter`, not the original SFT adapter.

In [42]:
RUN_FINAL_TEST = True

if RUN_FINAL_TEST:
    cfg = load_config(str(ACTIVE_CONFIG))
    cfg['run']['resolved_name'] = format_run_name(cfg['run']['name'], cfg)

    train_run_dir = make_run_dir(cfg['run']['output_root'], cfg['run']['resolved_name'])
    final_adapter = train_run_dir / 'final_adapter'

    if not final_adapter.exists():
        raise FileNotFoundError(f'Missing final adapter: {final_adapter}')

    cfg['model']['adapter_path'] = str(final_adapter)

    cfg['model']['tokenizer_name_or_path'] = cfg['model']['base_model_name_or_path']

    cfg['run']['name'] = cfg['run']['resolved_name'] + '_final_test'
    cfg['eval']['max_new_tokens'] = 256
    cfg['eval']['batch_size'] = 512

    eval_cfg = Path('/content/grpo_final_eval.yaml')
    eval_cfg.write_text(
        yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True),
        encoding='utf-8'
    )

    print(eval_cfg.read_text(encoding='utf-8')[:1500])

    !python RL_GRPO_train/train_grpo.py --config "{eval_cfg}" --eval-only --final-test


run:
  name: qwen25_3b_instruct_sft_grpo_g16_trainall_final_test
  output_root: RL_GRPO_train/outputs
  resolved_name: qwen25_3b_instruct_sft_grpo_g16_trainall
model:
  base_model_name_or_path: Qwen/Qwen2.5-3B-Instruct
  adapter_path: /content/drive/MyDrive/HPML_project/RL_GRPO_train/outputs/qwen25_3b_instruct_sft_grpo_g16_trainall/final_adapter
  tokenizer_name_or_path: Qwen/Qwen2.5-3B-Instruct
  trust_remote_code: true
  prompt_template: qwen_chat
  system_prompt: ''
  include_empty_system: false
  dtype: bf16
  device_map: null
  load_in_4bit: false
  is_trainable_adapter: true
train_dataset:
  kind: sft_train
  name: openai/gsm8k
  config: main
  path: SFT_train/data/gsm8k_sft_train.json
  limit: null
eval_dataset:
  kind: sft_val
  name: openai/gsm8k
  config: main
  path: SFT_train/data/gsm8k_sft_val.json
  limit: 725
final_eval_dataset:
  kind: gsm8k_test
  name: openai/gsm8k
  config: main
  start_index: 0
  limit: null
reward:
  answer_correct: 1.0
  answer_incorrect: 0.0
  fo